In [ ]:
# Import Libraries
import requests

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

import os
from os import path
import json
import time
import random
from pathlib import Path
from dotenv import load_dotenv
from scipy.spatial import cKDTree


In [ ]:
# Get file names 
hdb_points_file = "../datasets/resale_2019_to_2025_cleaned.geojson"
malls_points_file = "../datasets/singapore_malls_pois.csv"
output_file = "../cleaned_datasets/hdb_mall_distances.csv"

In [ ]:
load_dotenv()

url = "https://www.onemap.gov.sg/api/auth/post/getToken"

payload = {
    "email": os.getenv("ONEMAP_EMAIL"),
    "password": os.getenv("ONEMAP_EMAIL_PASSWORD")
}

response = requests.post(url, json=payload)

if response.status_code == 200:
    data = response.json()
    onemap_api_key = data.get("access_token")  
else:
    onemap_api_key = None
    print("Failed to get token")

### 1. Get names and coordinates of shopping malls

In [ ]:
# read file 
malls_df = pd.read_csv(malls_points_file)

# filter out rows where 'name' is null and select only the relevant columns
malls_df = malls_df[malls_df['name'].notna()][['name', 'lat', 'lon']].copy()

# remove duplicates based on 'name' and keep the first occurrence
malls_df = malls_df.drop_duplicates(subset='name', keep='first')

print(f"Remaining malls: {len(malls_df)}")
print(malls_df.head())

In [ ]:
# convert malls_df to a GeoDataFrame
malls_gdf = gpd.GeoDataFrame(
    malls_df,
    geometry=gpd.points_from_xy(malls_df['lon'], malls_df['lat']),
    crs="EPSG:4326"
)
# reproject to a projected CRS for accurate distance calculations 
malls_gdf = malls_gdf.to_crs(epsg=3414)
malls_gdf.head()

### 2. Get walking distance between HDB and closest mall 

3.1 Run a sample of postal codes to ensure code works, check the result output to see if there are any nulls 

In [ ]:
# --- 1. load and prep data ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
# take only the first 5 unique postals for this sample
postal_gdf_sample = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).head(5).copy()

malls_gdf = malls_gdf.to_crs(epsg=4326)

mall_coords = list(zip(malls_gdf.geometry.y, malls_gdf.geometry.x))
mall_names = malls_gdf['name'].tolist()

tree = cKDTree(mall_coords)

results = []

print(f"--- starting sample test for {len(postal_gdf_sample)} postals ---")

# --- 2. sample loop ---
for idx, row in postal_gdf_sample.iterrows():
    p_code = str(row['postal_code'])
    origin_coords = (row.geometry.y, row.geometry.x)
    
    print(f"\nchecking postal: {p_code} at {origin_coords}")
    
    # get 3 closest malls by straight line
    dist, indices = tree.query(origin_coords, k=3) 
    
    best_walk = float('inf')
    closest_mall = None

    for i in indices:
        dest_coords = mall_coords[i]
        mall_name = mall_names[i]
        
        # using the routingsvc endpoint from the docs
        url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
        
        try:

            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            headers = {"Authorization": token}
            # headers = {"Authorization": onemap_api_key}
            res = requests.get(url, headers=headers, timeout=10)
            
            if res.status_code == 200:
                data = res.json()
                if 'route_summary' in data:
                    walk_m = data['route_summary']['total_distance']
                    print(f"  -> found route to {mall_name}: {walk_m}m")
                    if walk_m < best_walk:
                        best_walk = walk_m
                        closest_mall = mall_name
                else:
                    print(f"  -> no route summary found for {mall_name}")
            else:
                print(f"  -> api error {res.status_code}: {res.text}")
            
            time.sleep(0.2) # slightly longer sleep for the sample
        except Exception as e:
            print(f"  -> request failed: {e}")

    results.append({
        "postal_code": p_code,
        "nearest_mall": closest_mall,
        "walking_dist_m": best_walk if best_walk != float('inf') else None
    })

# --- 3. display output ---
sample_df = pd.DataFrame(results)
print("\n--- final sample output ---")
print(sample_df)

# check if we actually got results
if sample_df['walking_dist_m'].isnull().all():
    print("\nwarning: all walking distances are null. check your api key and coordinate order.")
else:
    print("\nsuccess: distances found! you can now run the full script.")

3.2 Run full script if results generated for the sample postal codes

In [ ]:
# --- 1. Load Datasets (full) ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
postal_gdf = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).copy()

malls_gdf = malls_gdf.to_crs(epsg=4326)
mall_coords = list(zip(malls_gdf.geometry.y, malls_gdf.geometry.x))
mall_names = malls_gdf['name'].tolist()
tree = cKDTree(mall_coords)

# --- 2. Resume logic ---
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
    processed_postals = set(processed_df['postal_code'].astype(str))
    results = processed_df.to_dict('records')
    print(f"Resuming... {len(processed_postals)} postals already completed.")
else:
    results = []
    processed_postals = set()

# --- 3. Full loop ---
print(f"Starting processing for {len(postal_gdf) - len(processed_postals)} remaining postals...")

try:
    for idx, row in postal_gdf.iterrows():
        p_code = row['postal_code']
        if p_code in processed_postals:
            continue

        origin_coords = (row.geometry.y, row.geometry.x)
        dist, indices = tree.query(origin_coords, k=3)
        
        closest_walk = float('inf')
        closest_mall = None

        for i in indices:
            dest_coords = mall_coords[i]
            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
            
            try:
                res = requests.get(url, headers={"Authorization": token}, timeout=10)
                if res.status_code == 200:
                    data = res.json()
                    if 'route_summary' in data:
                        walk_m = data['route_summary']['total_distance']
                        if walk_m < closest_walk:
                            closest_walk = walk_m
                            closest_mall = mall_names[i]
                    else:
                        print(f"No route_summary for postal {p_code} -> mall {mall_names[i]}")
                else:
                    print(f"API error for postal {p_code} -> mall {mall_names[i]}: {res.status_code}")
                    print(res.text[:300])
                time.sleep(0.2)
            except:
                continue

        results.append({
            "postal_code": p_code,
            "nearest_mall": closest_mall,
            "walking_dist_m": closest_walk if closest_walk != float('inf') else None
        })
        processed_postals.add(p_code)

        if len(results) % 50 == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"progress: {len(results)} / {len(postal_gdf)} unique postals saved.")
            

except KeyboardInterrupt:
    print("\nPaused. Saving current progress...")

# Final save
pd.DataFrame(results).to_csv(output_file, index=False)
print(f"Task complete! Results saved to {output_file}")

In [ ]:
mall_distances = pd.read_csv("../cleaned_datasets/hdb_mall_distances.csv")
mall_distances.info()

### 5. Get feature variables of malls 
- num_malls_1km
- num_malls_2km

In [ ]:
# read files
unique_hdb = gpd.read_file(hdb_points_file).copy()
# malls_gdf -> malls dataframe
hdb_mall_distances = pd.read_csv("../cleaned_datasets/hdb_mall_distances.csv")

# get unique hdb locations
unique_hdb = unique_hdb.drop_duplicates(subset=["postal_code"]).copy()
unique_hdb = unique_hdb.to_crs(epsg=3414)
malls_gdf = malls_gdf.to_crs(epsg=3414)

# prepare unique hdb and mall coordinates for distance calculation
hdb_coords = np.array([(geom.x, geom.y) for geom in unique_hdb.geometry])
mall_coords = np.array([(geom.x, geom.y) for geom in malls_gdf.geometry])

mall_names = malls_gdf["name"].tolist()

# build cKDTree for efficient spatial queries
tree = cKDTree(mall_coords)

# get malls within 500m and 1km
indices_1km = tree.query_ball_point(hdb_coords, r=1000)
indices_2km = tree.query_ball_point(hdb_coords, r=2000)

# convert index arrays into mall name lists
unique_hdb["malls_within_1km"] = [
    [mall_names[i] for i in idx_list] for idx_list in indices_1km]

unique_hdb["malls_within_2km"] = [
    [mall_names[i] for i in idx_list] for idx_list in indices_2km]

# count columns
unique_hdb["num_malls_1km"] = unique_hdb["malls_within_1km"].apply(len)
unique_hdb["num_malls_2km"] = unique_hdb["malls_within_2km"].apply(len)

malls_counts = unique_hdb[[
    "postal_code",
    "num_malls_1km",
    "num_malls_2km"
]].copy()

print(malls_counts)

### 6. Merge with mall distance and conduct data integrity checks

In [ ]:
hdb_mall_distances = pd.read_csv("../cleaned_datasets/hdb_mall_distances.csv")
print(hdb_mall_distances.head())

In [ ]:
# convert postal_code to string for merging
hdb_mall_distances["postal_code"] = hdb_mall_distances["postal_code"].astype(str).str.zfill(6)
malls_counts["postal_code"] = malls_counts["postal_code"].astype(str).str.zfill(6)
hdb_mall_distances["nearest_mall"] = hdb_mall_distances["nearest_mall"].astype(str)

hdb_mall_final = hdb_mall_distances.merge(
    malls_counts,
    on="postal_code",
    how="left"
)

hdb_mall_final = hdb_mall_final[[
    "postal_code",
    "walking_dist_m",
    "num_malls_1km",
    "num_malls_2km"
]]

print(hdb_mall_final.head())
print(hdb_mall_final.info())

In [ ]:
# print cases where num_malls_1km is > num_malls_2km (should not happen)
print(hdb_mall_final[hdb_mall_final['num_malls_1km'] > hdb_mall_final['num_malls_2km']])
print(hdb_mall_distances['walking_dist_m'].isna())

In [ ]:
# Save file to csv 
hdb_mall_final.to_csv("../cleaned_datasets/hdb_mall_distances.csv", index=False)